# WP12 — Reward Hacking Detection & Robustness
**Prometheus v0.97**

Detects and mitigates reward hacking (Goodhart's law violations), reward
tampering, and specification gaming in learned reward functions:

1. **Goodhart detector** — proxy–true correlation monitoring; alerts when
   proxy climbs while true reward stagnates or falls
2. **Reward tampering detector** — weight-vector integrity and feature
   checksum verification
3. **Specification gaming detector** — VOR analysis, overoptimisation
   scoring, and hardcoding heuristics
4. **RobustRewardWrapper** — composes all three into a drop-in replacement
   for `ValueLearningAgent.get_reward()`
5. **Full benchmark** — 4 scenarios across 60 episodes

**References**: Krakovna et al. (2020); Gao et al. (2022); Pan et al. (2022);
Everitt et al. (2017)


In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    os.system('pip install scipy -q')
    os.system('git clone https://github.com/prometheus-ai/Prometheus_v0_PoC /content/Prometheus_v0_PoC 2>/dev/null || true')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.abspath('..'))

import warnings; warnings.filterwarnings('ignore')
print('Environment ready.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from prometheus.value_learning import ValueLearningAgent
from prometheus.reward_hacking import (
    GoodhartDetector,
    RewardTamperingDetector,
    SpecificationGamingDetector,
    RobustRewardWrapper,
)
from benchmarks.reward_hacking_benchmark import (
    RewardHackingBenchmark,
    _make_agent,
    N_FEATS,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

rng   = np.random.default_rng(42)
agent = _make_agent(seed=0)
print(f'ValueLearningAgent ready: {agent}')

---
## 1 — Goodhart Detector: Proxy vs True Reward

In [ ]:
det = GoodhartDetector(window=15, divergence_threshold=0.3, min_proxy_trend=0.005)

proxy_vals, true_vals, alerts = [], [], []

# Phase 1 (0-29): aligned
for i in range(30):
    v = float(i) * 0.05
    sig = det.update(v + rng.uniform(-0.02, 0.02),
                     v + rng.uniform(-0.02, 0.02))
    proxy_vals.append(sig.proxy_value)
    true_vals.append(sig.true_value)
    alerts.append(sig.diverging)

# Phase 2 (30-59): proxy rising, true falling (Goodhart!)
for i in range(30):
    proxy = 2.0 + float(i) * 0.2 + rng.uniform(-0.02, 0.02)
    true  = 1.5 - float(i) * 0.05 + rng.uniform(-0.02, 0.02)
    sig   = det.update(proxy, true)
    proxy_vals.append(sig.proxy_value)
    true_vals.append(sig.true_value)
    alerts.append(sig.diverging)

steps = list(range(60))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(steps, proxy_vals, 'b-', lw=1.5, label='Proxy reward')
ax1.plot(steps, true_vals,  'g-', lw=1.5, label='True reward')
ax1.axvline(30, color='orange', ls='--', lw=1.5, label='Divergence starts')
ax1.set_ylabel('Reward')
ax1.set_title('Goodhart Detector: Proxy vs True Reward', fontweight='bold')
ax1.legend()
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

ax2.fill_between(steps, [int(a) for a in alerts], alpha=0.6,
                 color='red', label='Goodhart alert')
ax2.axvline(30, color='orange', ls='--', lw=1.5)
ax2.set_ylabel('Alert')
ax2.set_xlabel('Step')
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['No', 'Yes'])
ax2.legend()
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()

stats = det.stats()
print(f"Warnings (phase 2): {sum(alerts[30:])}")
print(f"False alarms (phase 1): {sum(alerts[:30])}")
print(f"Last correlation: {stats['last_correlation']:.3f}")

---
## 2 — Reward Tampering Detector

In [ ]:
w_clean   = agent.get_weights()
w_norm    = float(np.linalg.norm(w_clean))
epsilons  = [0.0, 0.1, 0.3, 0.5, 0.8, 1.5, 3.0]
n_trials  = 30
det_rates = []

for eps in epsilons:
    hits = 0
    for _ in range(n_trials):
        td = RewardTamperingDetector(weight_change_threshold=0.5)
        td.register_weights(w_clean)
        noise = rng.standard_normal(w_clean.shape)
        noise *= (eps * w_norm) / max(float(np.linalg.norm(noise)), 1e-9)
        report = td.check_weights(w_clean + noise)
        if report.tampered:
            hits += 1
    det_rates.append(hits / n_trials)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71' if e < 0.6 else '#e74c3c' for e in epsilons]
bars = ax.bar([str(e) for e in epsilons], [r * 100 for r in det_rates],
              color=colors, edgecolor='white')
ax.axhline(50, color='gray', ls='--', lw=1)
for bar, v in zip(bars, det_rates):
    ax.text(bar.get_x() + bar.get_width()/2, v*100 + 1,
            f'{v*100:.0f}%', ha='center', fontsize=9)
ax.set_xlabel('Perturbation (ε × ||w||)')
ax.set_ylabel('Detection rate (%)')
ax.set_title('Reward Tampering Detection Rate vs Perturbation Magnitude',
             fontweight='bold')
green_patch = mpatches.Patch(color='#2ecc71', label='Clean (ε < 0.6)')
red_patch   = mpatches.Patch(color='#e74c3c', label='Tampered (ε ≥ 0.6)')
ax.legend(handles=[green_patch, red_patch])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

# Feature checksum demo
feats    = rng.standard_normal(N_FEATS)
checksum = RewardTamperingDetector.feature_checksum(feats)
injected = feats.copy(); injected[0] += 1e-6
td = RewardTamperingDetector()
r1 = td.check_features(feats,    checksum)
r2 = td.check_features(injected, checksum)
print(f'Clean features → tampered={r1.tampered} ({r1.description})')
print(f'Injected features → tampered={r2.tampered} ({r2.description})')

---
## 3 — Specification Gaming Detector

In [ ]:
det = SpecificationGamingDetector(vor_threshold=0.7, overopt_threshold=2.0,
                                   complexity_threshold=5.0)

cases = [
    ('Clean',           dict(proxy_reward=1.0, eval_reward=0.95,
                             replacement_reward=0.9, behaviour_complexity=50.0,
                             proxy_baseline=0.5, eval_baseline=0.5)),
    ('Overoptimised',   dict(proxy_reward=10.0, eval_reward=0.5,
                             proxy_baseline=0.5, eval_baseline=0.5)),
    ('Hardcoded',       dict(proxy_reward=15.0, behaviour_complexity=1.0)),
    ('Loophole (VOR)',  dict(proxy_reward=5.0,  replacement_reward=0.0)),
]

print(f'{"Case":<20} {"Gaming?":>8} {"Type":<20} {"VOR":>6} {"Overopt":>8} {"CplxR":>6}')
print('-' * 72)
results = []
for name, kwargs in cases:
    r = det.check(**kwargs)
    results.append(r)
    mark = '✓' if r.gaming_detected else '✗'
    print(f'{name:<20} {mark:>8} {r.gaming_type:<20} '
          f'{r.vor_score:>6.3f} {r.overopt_score:>8.3f} {r.complexity_ratio:>6.2f}')

# Bar chart: gaming scores
labels    = [c[0] for c in cases]
vor_vals  = [r.vor_score  for r in results]
oo_vals   = [r.overopt_score for r in results]
col_vals  = [r.complexity_ratio / 10 for r in results]  # scaled

x = np.arange(len(labels)); w = 0.25
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w,   vor_vals, w, label='VOR score',          color='#3498db')
ax.bar(x,       oo_vals,  w, label='Overopt score',      color='#e74c3c')
ax.bar(x + w,   col_vals, w, label='Complexity ratio /10',color='#9b59b6')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Score')
ax.set_title('Specification Gaming Scores by Attack Type', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

---
## 4 — RobustRewardWrapper: Full Pipeline

In [ ]:
wrapper = RobustRewardWrapper(
    base_agent    = agent,
    penalty_scale = 0.4,
)

w_clean = agent.get_weights().copy()
w_norm  = float(np.linalg.norm(w_clean))

test_cases = [
    ('Clean input',         lambda: wrapper.get_reward(rng.standard_normal(N_FEATS))),
    ('Weight tampered',     lambda: _tampered_verdict(wrapper, agent, w_clean, w_norm, rng)),
    ('Spec gaming (overopt)',lambda: wrapper.get_reward(
        rng.standard_normal(N_FEATS),
        eval_reward=wrapper.base_agent.get_reward(rng.standard_normal(N_FEATS)) * 0.05)),
    ('Feature injection',   lambda: _injection_verdict(wrapper, rng)),
]

def _tampered_verdict(wrapper, agent, w_clean, w_norm, rng):
    noise = rng.standard_normal(w_clean.shape)
    noise *= (2.0 * w_norm) / max(float(np.linalg.norm(noise)), 1e-9)
    agent.set_weights(w_clean + noise)
    v = wrapper.get_reward(rng.standard_normal(N_FEATS))
    agent.set_weights(w_clean)
    return v

def _injection_verdict(wrapper, rng):
    feats    = rng.standard_normal(N_FEATS)
    checksum = RewardTamperingDetector.feature_checksum(feats)
    injected = feats.copy(); injected[0] += 5.0
    return wrapper.get_reward(injected, reference_checksum=checksum)

print(f'{"Case":<28} {"Hacked?":>8} {"Penalty":>8} {"Adj reward":>12}')
print('-' * 62)
for name, fn in test_cases:
    verdict = fn()
    mark = '✓ DETECTED' if verdict.hacking_detected else '  clean'
    print(f'{name:<28} {mark:>10} {verdict.penalty_applied:>8.2f} '
          f'{verdict.reward:>12.4f}')
    if verdict.hacking_detected:
        print(f'  → {verdict.description[:70]}')

print(f'\nWrapper stats: {wrapper.stats()}')

---
## 5 — Full Benchmark: 4 Scenarios

In [ ]:
bench   = RewardHackingBenchmark(seed=42)
results = bench.run_all()
print(bench.summary_table(results))

In [ ]:
gh_r  = next(r for r in results if r.scenario == 'goodhart_scenario')
tp_r  = next(r for r in results if r.scenario == 'tampering_scenario')
gm_r  = next(r for r in results if r.scenario == 'gaming_scenario')
e2e_r = next(r for r in results if r.scenario == 'end_to_end')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# (A) Goodhart alerts
ax = axes[0]
bars = ax.bar(['Aligned\nphase', 'Diverging\nphase'],
              [gh_r.metrics['alerts_aligned_phase'],
               gh_r.metrics['alerts_diverging_phase']],
              color=['#2ecc71', '#e74c3c'], edgecolor='white')
for bar, v in zip(bars, [gh_r.metrics['alerts_aligned_phase'],
                          gh_r.metrics['alerts_diverging_phase']]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.2,
            str(v), ha='center', fontweight='bold')
ax.set_ylabel('Goodhart alerts')
ax.set_title('(A) Goodhart Detection', fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# (B) Tampering TPR/FPR
ax2 = axes[1]
eps_vals = tp_r.extra['epsilons'][1:]  # skip 0
tpr_vals = [tp_r.extra['tpr_by_epsilon'].get(e, 0.0) * 100 for e in eps_vals]
ax2.plot(eps_vals, tpr_vals, 'o-', color='#e74c3c', lw=2, ms=8)
ax2.axhline(50, color='gray', ls='--', lw=1)
ax2.set_xlabel('Perturbation ε')
ax2.set_ylabel('Detection rate (%)')
ax2.set_title('(B) Tampering TPR', fontweight='bold')
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# (C) Gaming detection rates
ax3 = axes[2]
bars3 = ax3.bar(['Clean\n(FPR)', 'Gaming\n(TPR)'],
                [gm_r.metrics['clean_false_positive']*100,
                 gm_r.metrics['gaming_detection_rate']*100],
                color=['#2ecc71', '#e74c3c'], edgecolor='white')
for bar, v in zip(bars3, [gm_r.metrics['clean_false_positive']*100,
                           gm_r.metrics['gaming_detection_rate']*100]):
    ax3.text(bar.get_x() + bar.get_width()/2, v + 1,
             f'{v:.0f}%', ha='center', fontweight='bold')
ax3.set_ylabel('Rate (%)')
ax3.set_title('(C) Spec Gaming TPR/FPR', fontweight='bold')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

# (D) End-to-end hacking rates
ax4 = axes[3]
bars4 = ax4.bar(['Clean\n(FPR)', 'Adversarial\n(TPR)'],
                [e2e_r.metrics['hacking_rate_clean']*100,
                 e2e_r.metrics['hacking_rate_adversarial']*100],
                color=['#2ecc71', '#e74c3c'], edgecolor='white')
for bar, v in zip(bars4, [e2e_r.metrics['hacking_rate_clean']*100,
                           e2e_r.metrics['hacking_rate_adversarial']*100]):
    ax4.text(bar.get_x() + bar.get_width()/2, v + 1,
             f'{v:.0f}%', ha='center', fontweight='bold')
ax4.set_ylabel('Hacking detection rate (%)')
ax4.set_title('(D) End-to-End Pipeline', fontweight='bold')
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

fig.suptitle('Reward Hacking Benchmark — Prometheus v0.97 (WP12)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
## Summary

| Property | Mechanism | Result |
|----------|-----------|--------|
| Goodhart detection | Rolling Pearson r + proxy trend slope | **alerts in diverging phase, silent on aligned** |
| Tampering detection | L2 weight delta + SHA-256 feature checksum | **TPR ≥ 75% for ε ≥ 0.6; FPR = 0%** |
| Spec gaming detection | VOR / overopt / complexity ratio | **gaming_rate ≥ 75%; clean FPR ≤ 10%** |
| End-to-end pipeline | All 3 detectors; penalty = n_flags × scale | **adversarial TPR ≥ 40%; clean FPR ≤ 15%** |
| Reward preservation | penalty_applied to proxy, not full zeroing | **clean reward preserved** |
| Interpretability | Every verdict has per-detector reports | **auditable** |

**Test coverage**: 92 tests, all passing (`pytest tests/test_reward_hacking.py -v`)

**Key design**:
- Three orthogonal detectors catch complementary failure modes
- `RobustRewardWrapper` is a drop-in for `ValueLearningAgent.get_reward()`
- Penalty is additive per detector flag, capped at 1.0 total
- `feature_checksum()` is SHA-256 byte-exact — any perturbation is caught

**Files**:
- `prometheus/reward_hacking.py` — GoodhartDetector, RewardTamperingDetector, SpecificationGamingDetector, RobustRewardWrapper
- `benchmarks/reward_hacking_benchmark.py` — 4-scenario benchmark (60 episodes)
- `tests/test_reward_hacking.py` — 92-test suite
- `notebooks/wp12_reward_hacking_demo.ipynb` — this notebook
